## 문제 1-1 : 기본 Chain 만들기 - AI 요리사

### GROQ 인증키 설정

 dotenv
 이 라이브러리는 민감한 정보(API 키, 비밀번호 등)를 소스 코드와 분리하여 .env 파일에 저장하고 안전하게 불러올 수 있게 돕습니다.

 load_dotenv(): 프로젝트 폴더의 .env 파일을 읽어서, 안에 적힌 GROQ_API_KEY=gsk_... 같은 줄을 환경변수로 등록합니다.
GROQ_API_KEY = os.getenv("GROQ_API_KEY"): 등록된 환경변수 중 GROQ_API_KEY 값을 꺼내 변수에 담습니다. 없으면 None이 들어갑니다.

In [1]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:3])

gsk


In [2]:
import langchain
print(langchain.__version__)

1.4.3


TEMPERATURE는 LLM이 답을 만들 때 얼마나 무작위로(창의적으로) 단어를 고를지 정하는 값

get_llm은 Groq 모델 객체를 만들어 주는 함수이다.
-> ChatOpenAI	이 함수가 돌려주는 값의 자료형

In [3]:
from langchain_openai import ChatOpenAI

# Groq(OpenAI 호환) 공통 설정 — 모델/온도 변경은 이곳만 수정
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트
GROQ_MODEL = "openai/gpt-oss-120b"  # 대안: "qwen/qwen3.8-27b"
#MOVIE_MODEL = "qwen/qwen3.8-27b"  # 영화 추천 예제용 
TEMPERATURE = 0.7 


def get_llm(model: str = GROQ_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """Groq 엔드포인트를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()
print(f'model name = {llm.model_name}')
# 잘 이해했는데 TEMPERATURE 호출해보기
print(f'temperature name = {llm.temperature}')

model name = openai/gpt-oss-120b
temperature name = 0.7


정식 정의
StrOutputParser는 langchain_core에 있는 출력 파서로, 채팅 모델이 반환한 메시지 객체에서 content(본문 텍스트)를 추출해 문자열로 돌려주는 Runnable입니다.

쉽게 말하면
택배 상자(AIMessage)를 뜯어서 안에 든 물건(답변 글자)만 꺼내 주는 사람입니다.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 프롬프트: AI에게 역할, 할 일, 답변 형식을 알려 줌
prompt = PromptTemplate.from_template("""
당신은 친절한 한식·양식 요리사입니다.
사용자가 가진 재료: {food}

위 재료로 만들 수 있는 요리 1개를 추천하고 간단한 레시피를 알려주세요.
없는 재료가 꼭 필요하면 기본 양념(소금, 기름 등) 정도만 추가하세요.

다음 형식으로 답변하세요.
{food}로 만들 수 있는 요리를 추천드립니다!

추천 요리: (요리 이름)
재료: (사용할 재료)
조리법:
1. (첫 번째 단계)
2. (두 번째 단계)
3. (세 번째 단계)
""")

# 2. 파서: AIMessage에서 글자만 꺼내 문자열로 바꿈
output_parser = StrOutputParser()

# 3. LCEL로 체인 연결: 프롬프트 → 모델 → 파서
chain = prompt | llm | output_parser

# 4. 사용자에게 재료 입력받기
food = input("재료를 입력하세요 (예: 계란, 밥, 김치): ")

# 5. 체인 실행 (스트리밍으로 글자가 흘러나오게 출력)
print(f"\n입력: {food}\n출력:")
for token in chain.stream({"food": food}):
    print(token, end="", flush=True)




입력: 계란, 밥 , 김치
출력:
계란, 밥 , 김치로 만들 수 있는 요리를 추천드립니다!

**추천 요리:** 김치 계란 볶음밥  

**재료:**  
- 밥 1공기  
- 김치 ½컵 (잘게 썬 것)  
- 계란 2개  
- 식용유 1큰술  
- 소금·후추 약간  
- (선택) 파가루 또는 파 조금, 참기름 약간  

**조리법:**  
1. **재료 준비** – 김치는 물기를 살짝 짜서 작게 다지고, 계란은 볼에 깨서 소금·후추를 약간 넣고 풀어 둡니다.  
2. **김치 볶기** – 팬에 식용유 ½큰술을 두르고 중불에 김치를 넣어 2~3분 정도 볶아 향을 내고, 김치가 살짝 투명해지면 밥을 넣고 고루 섞어가며 2분 정도 더 볶습니다.  
3. **계란 넣기** – 남은 식용유 ½큰술을 팬 가장자리에 두르고 풀어 둔 계란을 부어 반숙 정도가 될 때까지 살짝 저어준 뒤, 밥과 김치 위에 고루 퍼지도록 섞어 마무리합니다. 필요하면 소금·후추로 간을 맞추고, 원한다면 파가루와 참기름을 한두 방울 떨어뜨려 풍미를 더합니다.  

간단하고 든든한 한 끼 완성! 맛있게 드세요.

## 문제 1-2 : 2단계 Multi Chain 만들기 - 여행지 정보 시스템


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1단계 프롬프트: 여행지 → 명소 이름
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "관광 명소의 이름만 한 단어로 답하세요."),
    ("user", "{city}의 가장 대표적인 관광 명소 1곳을 추천해주세요."),
])

# 2단계 프롬프트: 명소 이름 → 상세 정보
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "정확하고 이해하기 쉽게 설명하세요."),
    ("user", "{spot}에 대해 아래 형식으로 알려주세요.\n\n"
             "명소: {spot}\n"
             "역사:\n1. ...\n2. ...\n\n"
             "특징:\n1. ...\n2. ...\n\n"
             "방문 팁:\n1. ...\n2. ..."),
])

# 각 단계 체인
chain1 = prompt1 | llm | StrOutputParser()   # 여행지 → 명소 이름
chain2 = prompt2 | llm | StrOutputParser()   # 명소 이름 → 상세 정보

# 두 체인을 LCEL로 연결
full_chain = (
    RunnablePassthrough.assign(spot=chain1)    # {"city"} → {"city", "spot"}
    | RunnablePassthrough.assign(info=chain2)  # → {"city", "spot", "info"}
)

# 실행
city = input("여행지를 입력하세요: ")
result = full_chain.invoke({"city": city})

# 각 단계 결과 출력
print("입력:", result["city"])
print("1단계 결과 (추천 명소):", result["spot"])
print("\n2단계 결과 (상세 정보):\n", result["info"])

입력: 일본
1단계 결과 (추천 명소): 후지산

2단계 결과 (상세 정보):
 **명소: 후지산**  

---

### 역사  
1. **신화와 전설** – 후지산은 고대 일본 신화에서 신성한 존재로 여겨졌으며, ‘아마테라스(천황)’와 같은 신들이 거주한다는 이야기가 전해집니다.  
2. **문화적 상징** – 에도 시대(1603‑1868)부터 화가·시인·목판화가들의 작품에 자주 등장해 일본의 대표적인 풍경으로 자리 잡았습니다.  
3. **산사와 순례** – 8세기경부터 산신 사원(예: 후지산 신사)이 건립돼 순례자들이 등산과 기도를 위해 찾았습니다.  
4. **근대 탐험** – 1860년대 서구 탐험가와 과학자들이 측량·지질 조사를 시작하면서 후지산은 세계적인 관심을 받게 되었습니다.  
5. **유네스코 세계문화유산** – 2013년 ‘후지산 – 신성한 풍경’으로 유네코 세계문화유산에 등재돼 자연·문화적 가치가 공식적으로 인정되었습니다.  

---

### 특징  
1. **높이와 형태** – 해발 3,776 m로 일본 최고봉이며, 완만한 원뿔형(화산 원뿔)으로 ‘완벽한 대칭’이 아름답게 유지됩니다.  
2. **활화산** – 마지막 분화는 1707년(호카이 분화)이며, 현재도 활화산으로 분류돼 관측소가 상시 운영됩니다.  
3. **네 계절의 모습** – 겨울엔 눈으로 뒤덮인 순백의 산, 봄엔 벚꽃과 함께, 여름엔 등산객이 몰려든 풍경, 가을엔 단풍과 구름 사이의 붉은 실루엣 등 사계절마다 색다른 매력을 보여줍니다.  
4. **다섯 호수(후지오코)** – 산 기슭에 위치한 가와구치, 마에, 야마나시, 사와, 하코네 등 다섯 호수는 사진 명소이자 레저 활동(보트, 온천) 중심지입니다.  
5. **문화와 예술** – ‘후지산을 그린 하쿠사이·우키요에’ 등 수많은 예술 작품에 영감을 주었으며, 현대에는 영화·음악·패션에서도 자주 등장합니다.  

---

### 방문 팁  
1. **등산 시즌** – 공식 등산 허가 기간은 7월 초 ~ 9월 초(대